In [ ]:
import os
from glob import glob
import geopandas
import pandas
import fiona
import numpy
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import rasterio
from itertools import chain
import networkx as nx

# from analysis_utils import *

In [ ]:
base_path = 'Z:\\jamaica\\Inputs'

In [ ]:
output_path = 'Z:\\jamaica\\Results'

In [ ]:
jamaica_crs = 3448

In [ ]:
jamaicaboundary = geopandas.read_file(os.path.join(base_path, 'jamaica.gpkg'))
jamaicaboundary = jamaicaboundary.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
jamaicaboundary['area_hectares'] = 0.0001*jamaicaboundary.geometry.area # Convert area to hectares
jamaica_total_area = jamaicaboundary['area_hectares'].sum()
#jamaicaboundary['area'] = jamaicaboundary.geometry.area 
#jamaica_total_area = jamaicaboundary['area'].sum()

In [ ]:
landcover = geopandas.read_file(os.path.join(base_path, '2013_landuse_landcover.gpkg'))[["OBJECTID","geometry","Classify"]]
landcover = landcover.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
landcover['area_hectares'] = 0.0001*landcover.geometry.area # Convert area to hectares

#landcover = geopandas.read_file(os.path.join(base_path, '2013_landuse_landcover.gpkg'))[["OBJECTID","geometry","Classify"]]
#landcover = landcover.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system

In [ ]:
bauxite = geopandas.read_file(os.path.join(base_path, 'nsmdb-bauxite_reserves.gpkg'))
bauxite = bauxite.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
bauxite['area_hectares'] = 0.0001*bauxite.geometry.area # Convert area to hectares
bauxite["total_area"]=bauxite.geometry.area.sum()

bauxite.rename(columns={"OBJECTID":"bauxite_id"},inplace=True)

In [ ]:
landcover_bauxite = bauxite[["bauxite_id","geometry"]] \
    .overlay(landcover.set_geometry("geometry"), how='intersection') #intersecting landcover with bauxite
landcover_bauxite = geopandas.GeoDataFrame(landcover_bauxite, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
landcover_bauxite.to_file(os.path.join(output_path, f'landcover_bauxite.gpkg'),driver="GPKG")

In [ ]:
landcover_bauxite_area = landcover_bauxite[['area_hectares', 'Classify']].groupby('Classify').sum()
print(landcover_bauxite_classified)
#landcover_bauxite_classified["area_percentage"] = 100.0*landcover_bauxite_classified["area_hectares_1"]/jamaica_total_area #to find out the percentage of landcover on bauxite


In [ ]:
landcover_bauxite_classified

In [ ]:
all_protected_areas = geopandas.read_file(os.path.join(base_path, 'allprotectedareas.gpkg'))
all_protected_areas = all_protected_areas.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
print(all_protected_areas)

In [ ]:
all_protected_areas['area_hectares'] = 0.0001*all_protected_areas.geometry.area # Convert area to hectares
all_protected_areas["total_area"]=all_protected_areas.geometry.area.sum()


landcover_bauxite_protected_areas = all_protected_areas[["LAYER","area_hectares"]] \
    .overlay(landcover_bauxite.set_geometry("geometry"), how='intersection') #intersecting landcover with bauxite


In [ ]:
layer_list = ['2013_landuse_landcover.gpkg','nsmdb-reefs_jan06_NEPA.gpkg','nsmdb-seagrass.gpkg', 'nsmdb-bauxite_reserves.gpkg', 'allprotectedareas.gpkg', 'new_bamboo.gpkg', 'MangrovesFN.gpkg', 'landcover_bauxite_allprotected.gpkg', 'hydrobasins.gpkg', 'Primary_Forest.gpkg', 'Fields_bare.gpkg', 'All_fields.gpkg', 'Plantations.gpkg', 'All_agriculture.gpkg', 'Wetland.gpkg', 'All_forest.gpkg', 'Secondary_Forest.gpkg', 'Dry_Forest.gpkg', 'Primary_Forest.gpkg']
layer_name = ["Landuse","Corals","Seagrass", "Bauxite", "All_Protected_Areas", "Bamboo", "FN_Mangroves", "Landcover_Protected_Bauxite", "Hydrobasins", "Primary_Forest", "Fields_bare", "All_fields", "Plantations", "All_agriculture", "Wetland", "All_forest", "Secondary_Forest", "Dry_Forest", "Primary_Forest"]
# layer_info = list(zip(layer_list,layer_name))
layer_info = dict(list(zip(layer_list,layer_name)))
area_outputs = []

In [ ]:
layer_info

In [ ]:
#for idx,(layer,layer_name) in enumerate(layer_info):
for layer,layer_name in layer_info.items():
    data = geopandas.read_file(os.path.join(base_path, layer))
    data = data.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
    data['area_hectares'] = 0.0001*data.geometry.area # Convert area to hectares
    total_area = data['area_hectares'].sum()
    area_outputs.append((layer_name,total_area,100.0*total_area/jamaica_total_area))
    if layer == '2013_landuse_landcover.gpkg':
        land_use_area = data[['area_hectares', 'Classify']].groupby('Classify').sum()
        land_use_area["area_percentage"] = 100.0*land_use_area["area_hectares"]/jamaica_total_area
        land_use_area.to_csv(os.path.join(output_path, '2013_landuse_landcover_areas2.csv'))
    

In [ ]:
area_outputs

In [ ]:
area_outputs = pandas.DataFrame(area_outputs,columns=["layer_name","area_hectares","area_percentage"])
area_outputs.to_csv(os.path.join(output_path, 'landuse_areas4.csv'))

In [ ]:
#Connectivity

In [ ]:
marine_layer_list = ['MangrovesFN.gpkg','nsmdb-reefs_jan06_NEPA.gpkg','nsmdb-seagrass.gpkg']
layer_name = ["FN_Mangroves","Corals","Seagrass"]
marine_layer_info = list(zip(layer_list,layer_name))
area_outputs = []

In [ ]:
for idx,(layer,marine_name) in enumerate(marine_layer_info):
    data = geopandas.read_file(os.path.join(base_path, layer))
    data = data.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
    for distance in [250, 500, 1000]:
        data[f"geometry_{distance}m_buffer"] = data.buffer(distance=distance)

In [ ]:
FN_Mangroves = geopandas.read_file(os.path.join(base_path, 'MangrovesFN.gpkg'))[["ID","geometry"]]
FN_Mangroves["total_area"]=FN_Mangroves.geometry.area
Corals = geopandas.read_file(os.path.join(base_path, 'nsmdb-reefs_jan06_NEPA.gpkg'))[["OBJECTID","geometry"]]
Corals.rename(columns={"OBJECTID":"corals_id"},inplace=True)
Seagrass = geopandas.read_file(os.path.join(base_path, 'nsmdb-seagrass.gpkg'))[["ID","geometry"]]
Seagrass.rename(columns={"ID":"seagrass_id"},inplace=True)
FN_Mangroves_copy = FN_Mangroves.copy()
FN_Mangroves.rename(columns={"ID":"mangrove_id"},inplace=True)
total_mangroves_area = FN_Mangroves["total_area"].sum()

In [ ]:
buffer_outputs = []
for distance in [250, 500, 1000]:
    FN_Mangroves_copy["geometry"] = FN_Mangroves_copy.buffer(distance=distance)
    Corals["geometry"] = Corals.buffer(distance=distance)
    Seagrass["geometry"] = Seagrass.buffer(distance=distance)
    mangroves_connectivity = FN_Mangroves_copy \
    .overlay(Corals.set_geometry("geometry"), how='intersection') \
    .overlay(Seagrass.set_geometry("geometry"), how='intersection')
    mangroves_connectivity = geopandas.GeoDataFrame(mangroves_connectivity, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    # print(mangroves_connectivity)
    mangroves_connectivity.to_file(os.path.join(output_path, f'mangroves_connectivity_{distance}m_buffer.gpkg'),driver="GPKG")
    mangroves_common = FN_Mangroves[["mangrove_id","geometry"]] \
    .overlay(mangroves_connectivity.set_geometry("geometry"), how='intersection')
    mangroves_common = geopandas.GeoDataFrame(mangroves_common, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
    mangroves_common = mangroves_common[mangroves_common["mangrove_id"] == mangroves_common["ID"]]
    mangroves_common = mangroves_common[["mangrove_id","geometry"]].dissolve(by='mangrove_id')
    mangroves_common["common_area"]=mangroves_common.geometry.area
    mangroves_common.to_file(os.path.join(output_path, f'mangroves_common_{distance}m_buffer.gpkg'),driver="GPKG")
    mangroves_common_area = mangroves_common["common_area"].sum()
    percentage_common = 100*mangroves_common_area/total_mangroves_area
    buffer_outputs.append((distance,mangroves_common_area,percentage_common))

buffer_outputs = pandas.DataFrame(buffer_outputs,columns=["distance_m","common_area","area_percentage"])
buffer_outputs.to_csv(os.path.join(output_path, 'buffer_outputs.csv'))

In [ ]:
landcover = geopandas.read_file(os.path.join(base_path, '2013_landuse_landcover.gpkg'))[["OBJECTID","geometry","Classify"]]
landcover = landcover.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
landcover['area_hectares'] = 0.0001*landcover.geometry.area # Convert area to hectares


In [ ]:
all_protected_areas = geopandas.read_file(os.path.join(base_path, 'allprotectedareas.gpkg'))
all_protected_areas = all_protected_areas.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
all_protected_areas['area_hectares'] = 0.0001*all_protected_areas.geometry.area # Convert area to hectares
all_protected_areas["total_area"]=all_protected_areas.geometry.area.sum()

In [ ]:
protected_areas = geopandas.read_file('Protected Sites/Protected Areas/Protected areas.shp')
    protected_areas = geopandas.read_file(
        "Protected Sites/Protected Areas/Protected areas.shp"
    )

    with_protected = with_forest_reserves
    for layer in protected_areas.LAYER.unique():
        with_protected = associate_vector(
            f'Protected Sites/Protected Areas/protected_areas_{layer}.gpkg',
            with_protected, cell,
            {'NAME':f'protected_area_{layer}_name'})
    with_protected['is_protected'] = (
        ~with_protected.protected_area_GAME_RESERVES_name.isna() |
        ~with_protected.protected_area_NATIONAL_PARK_name.isna() |
        ~with_protected.protected_area_PROTECTED_AREA_name.isna() |
        ~with_protected.protected_area_MARINE_PARK_name.isna()
            f"Protected Sites/Protected Areas/protected_areas_{layer}.gpkg",
            with_protected,
            cell,
            {"NAME": f"protected_area_{layer}_name"},
        )
    with_protected["is_protected"] = (
        ~with_protected.protected_area_GAME_RESERVES_name.isna()
        | ~with_protected.protected_area_NATIONAL_PARK_name.isna()
        | ~with_protected.protected_area_PROTECTED_AREA_name.isna()
        | ~with_protected.protected_area_MARINE_PARK_name.isna()
    )
    with_protected['is_proposed_protected'] = ~with_protected.protected_area_PROPOSED_PROTECTED_AREA_name.isna()
    with_protected[
        "is_proposed_protected"
    ] = ~with_protected.protected_area_PROPOSED_PROTECTED_AREA_name.isna()

In [ ]:
landcover_protected = all_protected_areas[['area_hectares']] \
    .overlay(landcover.set_geometry("geometry"), how='intersection') #intersecting landcover with bauxite
landcover_protected = geopandas.GeoDataFrame(landcover_protected, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
landcover_protected.to_file(os.path.join(output_path, f'landcover_protected.gpkg'),driver="GPKG")

In [ ]:
bauxite = geopandas.read_file(os.path.join(base_path, 'nsmdb-bauxite_reserves.gpkg'))
bauxite = bauxite.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
bauxite['area_hectares'] = 0.0001*bauxite.geometry.area # Convert area to hectares
bauxite["total_area"]=bauxite.geometry.area.sum()

bauxite.rename(columns={"OBJECTID":"bauxite_id"},inplace=True)

In [ ]:
landcover_bauxite = bauxite[["bauxite_id","geometry"]] \
    .overlay(landcover.set_geometry("geometry"), how='intersection') #intersecting landcover with bauxite
landcover_bauxite = geopandas.GeoDataFrame(landcover_bauxite, geometry="geometry",crs=f"EPSG:{jamaica_crs}")
landcover_bauxite.to_file(os.path.join(output_path, f'landcover_bauxite.gpkg'),driver="GPKG")

In [ ]:
landcover_bauxite_area = landcover_bauxite[['area_hectares', 'Classify']].groupby('Classify').sum()
print(landcover_bauxite_classified)
#landcover_bauxite_classified["area_percentage"] = 100.0*landcover_bauxite_classified["area_hectares_1"]/jamaica_total_area #to find out the percentage of landcover on bauxite


In [ ]:
landcover_bauxite_classified

In [ ]:
all_protected_areas = geopandas.read_file(os.path.join(base_path, 'allprotectedareas.gpkg'))
all_protected_areas = all_protected_areas.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system
print(all_protected_areas)

In [ ]:
all_protected_areas['area_hectares'] = 0.0001*all_protected_areas.geometry.area # Convert area to hectares
all_protected_areas["total_area"]=all_protected_areas.geometry.area.sum()


landcover_bauxite_protected_areas = all_protected_areas[["LAYER","area_hectares"]] \
    .overlay(landcover_bauxite.set_geometry("geometry"), how='intersection') #intersecting landcover with bauxite


In [ ]:
landcover10kgrid = geopandas.read_file(os.path.join(base_path, 'landcoversplit_10k.gpkg'))
landcover10kgrid = landcover10kgrid.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
landcover1kgrid = geopandas.read_file(os.path.join(base_path, 'landcoversplit_1k.gpkg'))
landcover1kgrid = landcover1kgrid.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system

In [ ]:
landcover1kgrid = pandas.DataFrame(landcover1kgrid,columns=["Classify","index_i"])
landcover1kgrid.to_csv(os.path.join(output_path, 'landcover1kgrid.csv'))
print(landcover1kgrid)

In [ ]:
forest_classes = [
    'Closed broadleaved forest (Primary Forest)',
    'Disturbed broadleaved forest (Secondary Forest)',
    'Secondary Forest',
    'Bamboo and Secondary Forest',
    'Fields and Secondary Forest',
    'Fields or Secondary Forest/Pine Plantation',
]

In [ ]:
forest = landcover[landcover["Classify"].isin(forest_classes)]
forest = forest.to_crs(epsg=jamaica_crs) # Covert geometry to Jamaica projection system

In [ ]:
forest.plot()

In [ ]:
forest_copy = forest.copy()
forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)

In [ ]:
for distance in [500, 1000, 5000, 10000]: 
    forest_copy = forest.copy()
    forest_copy.rename(columns={"OBJECTID":"forest_id"},inplace=True)
    forest_copy["geometry"] = forest_copy.geometry.buffer(distance=distance) #forest buffered different distances